# 07 – Backend Integration Test

**Purpose:** Simulate the FastAPI backend locally – load models, encoders, scaler, geo lookup, and test a full prediction pipeline.

**Steps:**
1. Load all trained models (Random Forest median + quantile NNs).
2. Load label encoders and scaler.
3. Load geo KDTree.
4. Define a helper function that takes (crop, season, state, lat, lon, fertilizer, pesticide, n, p, k, ph) → yield range.
5. Test on a few realistic inputs.

In [1]:
import numpy as np
import torch
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load Random Forest median
rf = joblib.load('../backend/models/yield_model_median_rf.pkl')

# Load quantile PyTorch models
class YieldPredictor(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 128), torch.nn.BatchNorm1d(128), torch.nn.ReLU(), torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 64), torch.nn.BatchNorm1d(64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 32), torch.nn.ReLU(),
            torch.nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

input_dim = 13  # adjust based on your feature engineering output (check X_train shape)
device = 'cpu'
model_lower = YieldPredictor(input_dim).to(device)
model_lower.load_state_dict(torch.load('../backend/models/yield_model_lower.pt', map_location=device))
model_lower.eval()
model_upper = YieldPredictor(input_dim).to(device)
model_upper.load_state_dict(torch.load('../backend/models/yield_model_upper.pt', map_location=device))
model_upper.eval()

# Load encoders and scaler
le_crop = joblib.load('../backend/models/le_crop.pkl')
le_season = joblib.load('../backend/models/le_season.pkl')
le_state = joblib.load('../backend/models/le_state.pkl')
scaler = joblib.load('../backend/models/scaler.pkl')

# Load geo lookup
geo_lookup = joblib.load('../backend/geo_lookup.pkl')
tree = geo_lookup['tree']
values = geo_lookup['values']

print("All components loaded.")

All components loaded.


In [2]:
def predict_yield_range(crop, season, state, lat, lon, fertilizer, pesticide, n, p, k, ph):
    # 1. Get rainfall & pH from geo lookup
    dist, idx = tree.query([[lon, lat]])
    rainfall, ph_geo = values[idx[0]]
    # Optional: use ph_geo instead of user input, but here we use user input as fallback
    
    # 2. Feature engineering (same as training)
    # This must exactly match how you built features in 02_feature_engineering
    # For simplicity, assume we have a function `prepare_features` that does:
    # - Fertilizer_per_ha, Pesticide_per_ha, log_Area (but Area not provided? Might need area from somewhere)
    # Actually your feature engineering used `Area` to compute per-hectare values. For live prediction,
    # the user would need to provide area, or you assume 1 hectare. We'll assume 1 hectare for demo.
    # This is a placeholder – you must replicate exactly the feature columns used in training.
    
    # Placeholder: create a feature vector with 13 numbers (order matters!)
    # You must load `X_train` to see the column order.
    
    # For now, I'll raise an error to remind you to implement this correctly.
    raise NotImplementedError("You must replicate the exact feature engineering from 02_feature_engineering.ipynb")

# Test with a sample input
test_input = {
    'crop': 'Rice',
    'season': 'Kharif',
    'state': 'Punjab',
    'lat': 30.9,
    'lon': 75.8,
    'fertilizer': 100,
    'pesticide': 10,
    'n': 20, 'p': 15, 'k': 25, 'ph': 7.0
}

# predict_yield_range(**test_input)

### Important

This notebook only verifies that models load. The actual feature extraction must **exactly mirror** the transformations in `02_feature_engineering.ipynb`. You will need to adapt the `predict_yield_range` function based on your exact feature order (e.g., load `X_train` columns).

For a quick test, you can also load a single row from `X_test.npy` and the corresponding original features to debug.